#### PLECS Automation: Interact through XML-RPC Interface
* XML-RPC portal should be enabled in PLECS settings
* I/O blocks should be properly added and built in the simulation model

In [1]:
# List values or ranges for parameters
import random
import numpy as np
import itertools

prelim = {"Vin": [200],
          "Vref": [160, 200, 240],
          "P": [100, 400, 700, 1000],
          "D1": np.arange(0.6, 1.02, 0.02).round(2).tolist(),
          "D2": np.arange(0.6, 1.02, 0.02).round(2).tolist(),
         }

# Generate all combinations for grid search
param_names = list(prelim.keys())
param_values = list(prelim.values())

# Calculate total number of combinations
total_combinations = 1
for values in param_values:
    total_combinations *= len(values)

# Generate all combinations
combinations = list(itertools.product(*param_values))

def get_a_param_set(prelim, index=0):
    """
    Get parameter set for grid search based on index.
    
    Args:
        prelim: Dictionary with parameter lists
        index: Current iteration index (0-based)
    
    Returns:
        Dictionary with parameter values for current iteration
    """
    # Check if index is valid
    if index >= total_combinations:
        raise ValueError(f"Index {index} exceeds total combinations ({total_combinations})")
    
    # Get the combination for current index
    current_combination = combinations[index]
    
    # Create parameter set dictionary
    param_set = {}
    for i, key in enumerate(param_names):
        param_set[key] = float(current_combination[i])
    
    return param_set

In [2]:
import threading
import numpy as np
import xmlrpc.client
import os
import pandas as pd
import csv
import time
import logging
import os
import shutil
import func_timeout
from func_timeout import func_set_timeout
from tqdm import tqdm


class PlecsThread(threading.Thread):
    
    opts = {'ModelVars': {'D1': 0.85, 'D2': 0.55, 'Vref': 200, 'P': 100, 'Ro': 200**2/100}}
    
    def __init__(self, threadID, name, model_name, result_path, num_req=1e4):
        super(PlecsThread, self).__init__()
        self.threadID = threadID
        self.name = name
        self.model_name = model_name
        self.num_req = num_req
        self.result_path = result_path + f"\\Performance.csv"
        self.result_path2 = result_path + f"\\Waveform"
        self.model_path = os.getcwd() + f"\\{self.model_name}.plecs"
        self.to_file()
        self.server = xmlrpc.client.Server('http://localhost:1080/RPC2"')
        
    def to_file(self):
        if not os.path.exists(self.result_path):
            columns = "idx Vin Vref P D1 D2 D0 Validity Pon_s1 Psw_s1 Pon_d1 \
            Pon_s2 Psw_s2 Pon_d2 Pon_s3 Psw_s3 Pon_d3 Pon_s4 Psw_s4 Pon_d4 \
            Pon_s5 Psw_s5 Pon_d5 Pon_s6 Psw_s6 Pon_d6 Pon_s7 Psw_s7 Pon_d7 Pon_s8 Psw_s8 Pon_d8 \
            Pcu Pfe Pc_esr ipk irms vp_t1 vp_t2 vp_t3 vp_t4 vs_t1 vs_t2 vs_t3 vs_t4 Vo"
            df = pd.DataFrame(columns=columns.split())
            df.to_csv(self.result_path, index=False)
            
    def load(self):
        self.server.plecs.load(self.model_path)
    
    def close(self):
        self.server.plecs.close(self.model_name)
    
    @func_set_timeout(60*3)
    def run_sim(self, opts):
        self.load()
        self.server.plecs.simulate(self.model_name, opts)
        self.close()
        
    def run(self):
        for i in tqdm(range(self.num_req)):
            self.loop(i)
    
    def loop(self, i):
        try:
            opts0 = get_a_param_set(prelim, i)
            opts = {'ModelVars': {}}
            opts['ModelVars'].update(opts0)
            opts['ModelVars']["Ro"] = round(opts0["Vref"]**2/opts0["P"], 2)
            print(opts)
            self.run_sim(opts)
            df = pd.read_csv(os.getcwd() + f"\\Single_Sim{self.threadID}.csv", header=None)
            with open(self.result_path, 'a', newline='') as f: 
                csv.writer(f).writerow(np.hstack(([i], list(get_a_param_set(prelim).values()), 
                                                  df.iloc[-1, 1:].values)))
            shutil.move("Single_Sim1_Wave.csv", os.path.join(self.result_path2, f"{i}.csv"))
        except Exception as e:
            print(e)

In [ ]:
import pandas as pd
import datetime

threads_num = 1
result_path = "D:\DAB" # any path you want
num_req = len(combinations)

start_time = datetime.datetime.now()
threads = []
for i in range(threads_num):
    thread = PlecsThread(i+1, f"{i+1}", f"DAB_sample{i+1}", result_path, num_req)
    threads.append(thread)
    thread.start()
for i in range(threads_num):
    threads[i].join()


  0%|          | 0/5292 [00:00<?, ?it/s]

{'ModelVars': {'Vin': 200.0, 'Vref': 160.0, 'P': 100.0, 'D1': 0.6, 'D2': 0.6, 'Ro': 256.0}}


  0%|          | 1/5292 [00:06<9:36:11,  6.53s/it]

{'ModelVars': {'Vin': 200.0, 'Vref': 160.0, 'P': 100.0, 'D1': 0.6, 'D2': 0.62, 'Ro': 256.0}}


  0%|          | 2/5292 [00:12<9:13:47,  6.28s/it]

...

100%|█████████▉| 5290/5292 [9:31:24<00:11,  5.85s/it]

{'ModelVars': {'Vin': 200.0, 'Vref': 240.0, 'P': 1000.0, 'D1': 1.0, 'D2': 0.98, 'Ro': 57.6}}


100%|█████████▉| 5291/5292 [9:31:30<00:05,  5.82s/it]

{'ModelVars': {'Vin': 200.0, 'Vref': 240.0, 'P': 1000.0, 'D1': 1.0, 'D2': 1.0, 'Ro': 57.6}}


100%|██████████| 5292/5292 [9:31:35<00:00,  6.48s/it]
